Linear Algebra 2 - Matrix Calculus and Positive Definite Matrices

Section 15: Matrix Calculus
Gradients, Jacobians, Hessians, and the chain rule in matrix notation.
This is the direct mathematical foundation of backpropagation.

Section 16: Positive Definite Matrices
Definition, tests, relationship to covariance matrices and convex optimization.
Why positive definiteness matters for loss landscape geometry.

Section 15 - Matrix Calculus

Matrix calculus extends derivatives to vectors and matrices.

Gradient of f(x): df/dx, a vector of partial derivatives. Shape same as x.
Jacobian of f(x): df/dx where f is a vector function. Shape (m, n) for f: R^n -> R^m.
Hessian of f(x): d^2f/dx^2, the matrix of second derivatives. Shape (n, n).

Common results used everywhere in ML:
- gradient of (Ax - b)^2 with respect to x: 2 A^T (Ax - b)
- gradient of x^T A x with respect to x: (A + A^T) x   (2Ax if A is symmetric)
- gradient of log sigmoid(x): 1 - sigmoid(x)

In [ ]:
# Base Python: numerical gradient via central finite differences
# df/dx_i ≈ (f(x + eps*e_i) - f(x - eps*e_i)) / (2*eps)

def numerical_gradient(f, x, eps=1e-5):
    grad = [0.0] * len(x)
    for i in range(len(x)):
        x_plus  = x[:]; x_plus[i]  += eps
        x_minus = x[:]; x_minus[i] -= eps
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * eps)
    return grad

# f(x) = x0^2 + 3*x1 + x0*x1  ->  grad = [2*x0 + x1,  3 + x0]
def f(x):
    return x[0]**2 + 3*x[1] + x[0]*x[1]

x = [2.0, 4.0]
grad_num  = numerical_gradient(f, x)
grad_anal = [2*x[0] + x[1], 3 + x[0]]  # analytical

print(f'x            = {x}')
print(f'numerical    = {[round(g, 6) for g in grad_num]}')
print(f'analytical   = {grad_anal}')

In [ ]:
import numpy as np

# analytical gradients of common ML loss functions

# MSE loss: L = (1/n) ||Xw - y||^2
# gradient dL/dw = (2/n) X^T (Xw - y)
np.random.seed(7)
n, d = 50, 4
X = np.random.randn(n, d)
y = np.random.randn(n)
w = np.random.randn(d)

residual   = X @ w - y
mse_loss   = (residual**2).mean()
grad_mse   = (2 / n) * X.T @ residual
print(f'MSE loss: {mse_loss:.4f}')
print(f'grad_MSE: {grad_mse.round(4)}')

# gradient of L2 regularisation: d/dw (lambda * ||w||^2) = 2 * lambda * w
lam = 0.01
grad_l2  = 2 * lam * w
grad_total = grad_mse + grad_l2
print(f'\nL2 reg grad: {grad_l2.round(6)}')
print(f'total grad : {grad_total.round(4)}')

# numerical verification
def loss(w_):
    return ((X @ w_ - y)**2).mean() + lam * (w_**2).sum()

eps = 1e-5
grad_numerical = np.array([(loss(w + eps*np.eye(d)[i]) - loss(w - eps*np.eye(d)[i])) / (2*eps)
                            for i in range(d)])
print(f'\nnumerical grad: {grad_numerical.round(4)}')
print(f'match: {np.allclose(grad_total, grad_numerical, atol=1e-5)}')

In [ ]:
import torch

# PyTorch: Jacobian of a vector function f: R^3 -> R^2
# f(x) = [x0^2 + x1,  x1*x2 + x0]
# J[i,j] = df_i / dx_j

def f_vec(x):
    return torch.stack([x[0]**2 + x[1],
                        x[1] * x[2] + x[0]])

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
J = torch.autograd.functional.jacobian(f_vec, x)

print(f'Jacobian shape: {J.shape}  (2 outputs, 3 inputs)')
print(f'Jacobian:\n{J}')

# analytical Jacobian at x=[1,2,3]:
# row 0: [2*x0, 1, 0] = [2, 1, 0]
# row 1: [1, x2, x1]  = [1, 3, 2]
J_anal = torch.tensor([[2.0, 1.0, 0.0],
                        [1.0, 3.0, 2.0]])
print(f'\nanalytical:\n{J_anal}')
print(f'match: {torch.allclose(J, J_anal)}')

In [ ]:
import torch

# Hessian: matrix of second-order partial derivatives
# H[i,j] = d^2f / (dx_i dx_j)

def f_scalar(x):
    # f(x) = x0^2 + 2*x1^2 + x0*x1
    # H = [[2, 1], [1, 4]]  (constant, independent of x)
    return x[0]**2 + 2*x[1]**2 + x[0]*x[1]

x = torch.tensor([3.0, -1.0])
H = torch.autograd.functional.hessian(f_scalar, x)

print(f'Hessian:\n{H}')
H_anal = torch.tensor([[2.0, 1.0],
                        [1.0, 4.0]])
print(f'analytical:\n{H_anal}')
print(f'match: {torch.allclose(H, H_anal)}')

# eigenvalues of H tell us about curvature of f at x
eigvals = torch.linalg.eigvalsh(H)
print(f'\nHessian eigenvalues: {eigvals}  (all positive = locally convex at x)')

In [ ]:
import tensorflow as tf
import numpy as np

# TensorFlow: gradient and Jacobian via GradientTape

X_np = np.random.randn(20, 3).astype(np.float32)
y_np = np.random.randn(20).astype(np.float32)
X_tf = tf.constant(X_np)
y_tf = tf.constant(y_np)

w = tf.Variable(tf.zeros(3))

with tf.GradientTape() as tape:
    residual = tf.linalg.matvec(X_tf, w) - y_tf
    loss = tf.reduce_mean(residual**2)

grad = tape.gradient(loss, w)
print(f'TF MSE loss: {loss.numpy():.4f}')
print(f'TF gradient: {grad.numpy().round(4)}')

# analytical check
grad_anal = (2 / len(y_np)) * X_np.T @ (X_np @ w.numpy() - y_np)
print(f'analytical:  {grad_anal.round(4)}')
print(f'match: {np.allclose(grad.numpy(), grad_anal, atol=1e-5)}')

In [ ]:
import numpy as np

# Chain rule through a linear layer followed by MSE loss
# Forward:  z = W @ x + b,  loss = ||z - y||^2
# Backward: dL/dW = dL/dz * dz/dW = 2*(z - y) * x^T
#           dL/db = 2*(z - y)

np.random.seed(1)
d_in, d_out = 4, 2
W = np.random.randn(d_out, d_in)
b = np.random.randn(d_out)
x = np.random.randn(d_in)
y = np.array([1.0, -1.0])

# forward pass
z    = W @ x + b
loss = np.sum((z - y)**2)

# backward pass
dL_dz = 2 * (z - y)           # shape (d_out,)
dL_dW = np.outer(dL_dz, x)    # shape (d_out, d_in)
dL_db = dL_dz                  # shape (d_out,)
dL_dx = W.T @ dL_dz            # shape (d_in,) -- gradient wrt input

print(f'loss: {loss:.4f}')
print(f'dL/dW shape: {dL_dW.shape}')
print(f'dL/dW:\n{dL_dW.round(4)}')
print(f'dL/db: {dL_db.round(4)}')
print(f'dL/dx: {dL_dx.round(4)}')

Section 16 - Positive Definite Matrices

A symmetric matrix A is positive definite (PD) if x^T A x > 0 for every nonzero vector x.
It is positive semi-definite (PSD) if x^T A x >= 0.

Three equivalent tests for positive definiteness:
1. All eigenvalues are strictly positive
2. Cholesky decomposition exists (no failure)
3. All leading principal minors have positive determinant

In ML:
- Covariance matrices are always PSD (PD when full rank)
- The Hessian of a convex loss is PD everywhere -- gradient descent converges
- Gram matrices K[i,j] = k(x_i, x_j) in kernel methods are PSD by definition
- A PD matrix always has a unique inverse

In [ ]:
# Base Python: test positive definiteness via Sylvester's criterion
# A is PD iff all leading principal submatrices have positive determinant

def det_2x2(M):
    return M[0][0]*M[1][1] - M[0][1]*M[1][0]

def det_3x3(M):
    return (M[0][0]*(M[1][1]*M[2][2] - M[1][2]*M[2][1])
          - M[0][1]*(M[1][0]*M[2][2] - M[1][2]*M[2][0])
          + M[0][2]*(M[1][0]*M[2][1] - M[1][1]*M[2][0]))

def is_positive_definite_3x3(M):
    d1 = M[0][0]
    d2 = det_2x2([[M[0][0], M[0][1]], [M[1][0], M[1][1]]])
    d3 = det_3x3(M)
    return d1 > 0 and d2 > 0 and d3 > 0

A_pd = [[4.0, 2.0, 1.0],
        [2.0, 3.0, 1.0],
        [1.0, 1.0, 2.0]]

A_not_pd = [[1.0, 2.0, 0.0],
             [2.0, 1.0, 0.0],
             [0.0, 0.0, 1.0]]

print(f'A_pd is positive definite: {is_positive_definite_3x3(A_pd)}')
print(f'A_not_pd is PD:           {is_positive_definite_3x3(A_not_pd)}')

In [ ]:
import numpy as np

def check_pd_numpy(A, label):
    eigenvalues = np.linalg.eigvalsh(A)  # eigvalsh is for symmetric matrices
    is_pd_eigen = bool(np.all(eigenvalues > 0))
    try:
        np.linalg.cholesky(A)
        is_pd_chol = True
    except np.linalg.LinAlgError:
        is_pd_chol = False
    print(f'{label}:')
    print(f'  eigenvalues: {eigenvalues.round(4)}')
    print(f'  PD by eigenvalues: {is_pd_eigen}')
    print(f'  PD by Cholesky:    {is_pd_chol}')

A_pd = np.array([[4.0, 2.0, 1.0],
                 [2.0, 3.0, 1.0],
                 [1.0, 1.0, 2.0]])

A_psd = np.array([[1.0, 1.0],
                  [1.0, 1.0]])  # eigenvalues: 0, 2  --> PSD not PD

A_ind = np.array([[1.0, 2.0],
                  [2.0, 1.0]])  # eigenvalues: -1, 3 --> indefinite

check_pd_numpy(A_pd,  'A_pd  (positive definite)')
print()
check_pd_numpy(A_psd, 'A_psd (positive semi-definite)')
print()
check_pd_numpy(A_ind, 'A_ind (indefinite)')

In [ ]:
import torch

A_pd = torch.tensor([[4.0, 2.0, 1.0],
                     [2.0, 3.0, 1.0],
                     [1.0, 1.0, 2.0]])

# eigenvalues via torch.linalg.eigvalsh (symmetric/hermitian)
eigvals = torch.linalg.eigvalsh(A_pd)
print(f'eigenvalues: {eigvals}')
print(f'all positive: {bool(torch.all(eigvals > 0))}')

# Cholesky test
try:
    L = torch.linalg.cholesky(A_pd)
    print(f'Cholesky succeeded: PD confirmed')
    print(f'L:\n{L.round(4)}')
    print(f'L @ L.T = A: {torch.allclose(L @ L.T, A_pd, atol=1e-5)}')
except RuntimeError:
    print('Cholesky failed: not PD')

In [ ]:
import tensorflow as tf
import numpy as np

A_pd = tf.constant([[4.0, 2.0, 1.0],
                    [2.0, 3.0, 1.0],
                    [1.0, 1.0, 2.0]])

L = tf.linalg.cholesky(A_pd)
print(f'TF Cholesky L:\n{L.numpy().round(4)}')
print(f'L @ L.T = A: {np.allclose((L @ tf.transpose(L)).numpy(), A_pd.numpy(), atol=1e-5)}')

In [ ]:
import numpy as np

# ML example: covariance matrix is always PSD, kernel matrix is PSD

np.random.seed(3)
X = np.random.randn(30, 5)  # 30 samples, 5 features

# sample covariance matrix: always PSD
X_c = X - X.mean(axis=0)
C   = (1 / (X.shape[0] - 1)) * X_c.T @ X_c
eigvals_C = np.linalg.eigvalsh(C)
print(f'covariance eigenvalues: {eigvals_C.round(4)}  (all >= 0)')
print(f'covariance is PSD: {bool(np.all(eigvals_C >= -1e-10))}')

# RBF kernel matrix: K[i,j] = exp(-||x_i - x_j||^2 / (2*sigma^2))
# always PSD by Mercer's theorem
sigma = 1.0
X_small = X[:8]
dists_sq = np.sum((X_small[:, None, :] - X_small[None, :, :])**2, axis=-1)
K = np.exp(-dists_sq / (2 * sigma**2))
eigvals_K = np.linalg.eigvalsh(K)
print(f'\nRBF kernel eigenvalues: {eigvals_K.round(6)}  (all >= 0)')
print(f'kernel is PSD: {bool(np.all(eigvals_K >= -1e-10))}')

# Hessian of MSE (X^T X) is PSD; PD when X has full column rank
H_mse = X.T @ X  # proportional to Hessian of MSE
eigvals_H = np.linalg.eigvalsh(H_mse)
print(f'\nHessian eigenvalues: {eigvals_H.round(4)}  (all positive -> convex loss)')